Download graphs listed on [ECL](https://userweb.cs.txstate.edu/~burtscher/research/ECLgraph/) as ready **SuiteSparse** Matrix Market `.mtx` ([sparse.tamu.edu](https://sparse.tamu.edu/)): map by `origin` (SSMC/SNAP/DIMACS10). Unpack MM `.tar.gz` → `boruvka/graphs`. On error or timeout, skip. No straightforward collection entry for: Galois (`*.sym.egr`), `internet.egr`, `USA-road-d.NY.egr`; full USA road graph → `GAP/GAP-road`.

In [5]:
import io, ssl, tarfile, urllib.request
from pathlib import Path

# macOS / some Python builds lack cert bundle → CERTIFICATE_VERIFY_FAILED.
# Unverified HTTPS (MITM risk). Prefer: run `/Applications/Python .../Install Certificates.command` or use certifi.
_SSL = ssl._create_unverified_context()

BASE = "https://suitesparse-collection-website.herokuapp.com/MM"
OUT = (Path("..") / "boruvka" / "graphs").resolve()
OUT.mkdir(parents=True, exist_ok=True)

# out stem (ECL basename without .egr) → (SuiteSparse group, matrix name, ECL origin column).
GRAPHS = [
    ("amazon0601", "SNAP", "amazon0601", "SNAP"),
    ("citationCiteseer", "DIMACS10", "citationCiteseer", "SSMC"),
    ("as-skitter", "SNAP", "as-Skitter", "SNAP"),
    ("coPapersDBLP", "DIMACS10", "coPapersDBLP", "SSMC"),
    ("in-2004", "LAW", "in-2004", "SSMC"),
    ("soc-LiveJournal1", "SNAP", "soc-LiveJournal1", "SNAP"),
    ("cit-Patents", "SNAP", "cit-Patents", "SSMC"),
    ("USA-road-d.USA", "DIMACS10", "road_usa", "Dimacs"),
    ("delaunay_n24", "DIMACS10", "delaunay_n24", "SSMC"),
    ("kron_g500-logn21", "DIMACS10", "kron_g500-logn21", "SSMC"),
    ("europe_osm", "DIMACS10", "europe_osm", "SSMC"),
    # ("uk-2002", "LAW", "uk-2002", "SSMC"), too large
]


def fetch_mtx(stem: str, group: str, name: str, timeout: int = 600) -> None:
    dest = OUT / f"{stem}.mtx"
    if dest.is_file():
        print("SKIP", stem, "already exists")
        return
    url = f"{BASE}/{group}/{name}.tar.gz"
    print(stem, "←", url)
    with urllib.request.urlopen(url, timeout=timeout, context=_SSL) as r:
        data = r.read()
    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tf:
        members = [m for m in tf.getmembers() if m.name.endswith(".mtx")]
        if not members:
            raise FileNotFoundError("no .mtx in archive")
        m = max(members, key=lambda x: x.size or 0)
        f = tf.extractfile(m)
        if f is None:
            raise RuntimeError("extractfile")
        dest.write_bytes(f.read())
    print("OK", stem)


for stem, grp, mtx, tc_origin in GRAPHS:
    try:
        fetch_mtx(stem, grp, mtx)
    except Exception as e:
        print("SKIP", stem, e)

SKIP amazon0601 already exists
SKIP citationCiteseer already exists
SKIP as-skitter already exists
SKIP coPapersDBLP already exists
SKIP in-2004 already exists
SKIP soc-LiveJournal1 already exists
SKIP cit-Patents already exists
SKIP USA-road-d.USA already exists
SKIP delaunay_n24 already exists
SKIP kron_g500-logn21 already exists
SKIP europe_osm already exists


In [6]:
# uses GRAPHS, OUT from the cell above
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.io import mmread
from scipy.sparse import csgraph


def _mm_hdr(path):
    b = None
    with open(path, encoding="latin-1", errors="replace") as f:
        for ln in f:
            s = ln.strip()
            if s.startswith("%%MatrixMarket"):
                b = s.split()
            elif s and not s.startswith("%"):
                n, m = map(int, s.replace(",", " ").split()[:2])
                break
    if not b or len(b) < 5:
        raise ValueError("bad MatrixMarket header")
    return n, m, b[-2].lower(), b[-1].lower()


def _row(stem, mtx_path=None):
    p = Path(mtx_path) if mtx_path is not None else OUT / f"{stem}.mtx"
    fn = p.name
    bad = {"file": fn, "graph_kind": "", "|V|": "", "|E|": "", "components_weak": "", "weights_nonneg": "", "note": ""}
    if not p.is_file() or p.stat().st_size == 0:
        return {**bad, "note": "missing or empty"}
    n, m, fld, sym = _mm_hdr(p)
    u = sym in ("symmetric", "hermitian")
    A = mmread(p).tocsr()
    A.eliminate_zeros()
    if n != m or A.shape[0] != n:
        return {**bad, "note": "not square"}
    co = A.tocoo()
    r, c, d = co.row, co.col, co.data
    E = int(((r < c) | ((r == c) & (d != 0))).sum()) if u else int((d != 0).sum())
    B = A.astype(bool) if u else (A + A.T).astype(bool)
    comp = csgraph.connected_components(B, directed=False)[0]
    if fld == "pattern":
        w = "n/a (pattern)"
    elif np.iscomplexobj(d):
        w = "yes" if np.all(np.real(d) >= 0) else "no"
    else:
        w = "yes" if np.all(d >= 0) else "no"
    return {"file": fn, "graph_kind": "undirected" if u else "directed", "|V|": n, "|E|": E, "components_weak": comp, "weights_nonneg": w, "note": ""}


tbl = []
for s, *_ in GRAPHS:
    try:
        tbl.append(_row(s))
    except Exception as e:
        tbl.append({"file": f"{s}.mtx", "graph_kind": "", "|V|": "", "|E|": "", "components_weak": "", "weights_nonneg": "", "note": str(e)})
display(pd.DataFrame(tbl))


,file,graph_kind,|V|,|E|,components_weak,weights_nonneg,note
0,amazon0601.mtx,undirected,403394,2443408,7,yes,
1,citationCiteseer.mtx,undirected,268495,1156647,1,yes,
2,as-skitter.mtx,undirected,1696415,11095298,756,yes,
3,coPapersDBLP.mtx,undirected,540486,15245729,1,yes,
4,in-2004.mtx,undirected,1382908,13591473,134,yes,
5,soc-LiveJournal1.mtx,undirected,4847571,42851237,1876,yes,
6,cit-Patents.mtx,undirected,3774768,16518947,3627,yes,
7,USA-road-d.USA.mtx,undirected,23947347,28854312,1,yes,
8,delaunay_n24.mtx,undirected,16777216,50331601,1,yes,
9,kron_g500-logn21.mtx,undirected,2097152,91042010,553159,yes,


In [7]:
# Skip weights_nonneg=="no" (как в таблице ячейки 2); несимметричные → min-симметризация; перезапись <stem>.mtx → GRAPHS_BENCH.

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.io import mmread, mmwrite
from scipy.sparse import coo_matrix, issparse


def _symmetrize_min(A):
    A = A.tocoo()
    n, best = A.shape[0], {}
    for i, j, v in zip(A.row, A.col, A.data):
        if i == j:
            continue
        a, b = (i, j) if i < j else (j, i)
        t = best.get((a, b))
        if t is None or v < t:
            best[(a, b)] = v
    if not best:
        return coo_matrix((n, n)).tocsr()
    rs, cs, ds = [], [], []
    for (a, b), v in best.items():
        rs += (a, b)
        cs += (b, a)
        ds += (v, v)
    return coo_matrix((ds, (rs, cs)), shape=(n, n)).tocsr()


def _fail(row, note):
    row["bench"] = False
    row["note"] = note
    return row


GRAPHS_BENCH, bench_log = [], []

for g in GRAPHS:
    stem = g[0]
    p = OUT / f"{stem}.mtx"
    try:
        info = _row(stem)
    except Exception as e:
        bench_log.append(_fail({"file": f"{stem}.mtx"}, str(e)))
        continue
    if info["weights_nonneg"] == "no":
        bench_log.append(_fail(dict(info), "negative weights (table)"))
        continue
    if not p.is_file() or not p.stat().st_size:
        bench_log.append(_fail({"file": p.name}, "missing"))
        continue
    try:
        n0, m0, _, sym = _mm_hdr(p)
        raw = mmread(p)
        co = raw.tocoo() if issparse(raw) else coo_matrix(raw)
        if co.nnz and np.iscomplexobj(co.data):
            raise ValueError("complex weights")
        A = co.tocsr().astype(np.float64)
        A.eliminate_zeros()
        if n0 != m0 or A.shape[0] != n0:
            raise ValueError("non-square")
        symmd = sym not in ("symmetric", "hermitian")
        if symmd:
            A = _symmetrize_min(A)
            A.eliminate_zeros()
        if A.nnz and np.any(A.data < 0):
            bench_log.append(_fail({"file": p.name}, "negative after symmetrize"))
            continue
        mmwrite(str(p), A.astype(np.float64), field="real", symmetry="symmetric")
        GRAPHS_BENCH.append(g)
        bench_log.append({**_row(stem, mtx_path=p), "bench": True, "prep": "sym" if symmd else "—"})
    except Exception as e:
        bench_log.append(_fail({"file": p.name}, str(e)))

display(pd.DataFrame(bench_log))


,file,graph_kind,|V|,|E|,components_weak,weights_nonneg,note,bench,prep
0,amazon0601.mtx,undirected,403394,2443408,7,yes,,True,—
1,citationCiteseer.mtx,undirected,268495,1156647,1,yes,,True,—
2,as-skitter.mtx,undirected,1696415,11095298,756,yes,,True,—
3,coPapersDBLP.mtx,undirected,540486,15245729,1,yes,,True,—
4,in-2004.mtx,undirected,1382908,13591473,134,yes,,True,—
5,soc-LiveJournal1.mtx,undirected,4847571,42851237,1876,yes,,True,—
6,cit-Patents.mtx,undirected,3774768,16518947,3627,yes,,True,—
7,USA-road-d.USA.mtx,undirected,23947347,28854312,1,yes,,True,—
8,delaunay_n24.mtx,undirected,16777216,50331601,1,yes,,True,—
9,kron_g500-logn21.mtx,undirected,2097152,91042010,553159,yes,,True,—


In [8]:
# boruvka_spla; CSV -> experiments/results/spla-<graph>.csv; run after prep cell (GRAPHS_BENCH, предобработанные <stem>.mtx).
import subprocess
from subprocess import TimeoutExpired

import pandas as pd
from IPython.display import display
from pathlib import Path

EX = (Path("..") / "boruvka" / "boruvka_spla" / "build" / "boruvka_spla").resolve()
RESULTS = (Path(".") / "results").resolve()
RESULTS.mkdir(parents=True, exist_ok=True)
TIMEOUT_S = 240

if not EX.is_file():
    bench = [{"graph": s, "status": "error", "time_ms": None, "mst_weight": None, "mst_vertices": None, "mst_edges": None, "note": f"no binary: {EX}"} for s, *_ in GRAPHS_BENCH]
else:
    bench = []
    for stem, *_ in GRAPHS_BENCH:
        mtx = OUT / f"{stem}.mtx"
        out_csv = RESULTS / f"spla-{stem}.csv"
        rec = {"graph": stem, "status": "", "time_ms": None, "mst_weight": None, "mst_vertices": None, "mst_edges": None, "note": ""}
        if not mtx.is_file() or mtx.stat().st_size == 0:
            rec["status"] = "skip"
            rec["note"] = "missing .mtx"
            bench.append(rec)
            continue
        cmd = [str(EX), "--mtxpath", str(mtx), "--out", str(out_csv), "--niters", "1", "--warmup", "0"]
        try:
            p = subprocess.run(cmd, capture_output=True, text=True, timeout=TIMEOUT_S)
        except TimeoutExpired:
            rec["status"] = "timeout"
            rec["note"] = ""
            bench.append(rec)
            continue
        if p.returncode != 0:
            rec["status"] = "error"
            rec["note"] = (p.stderr or p.stdout or "").strip()[:800]
            bench.append(rec)
            continue
        try:
            row = pd.read_csv(out_csv).iloc[0]
            rec["status"] = "ok"
            rec["time_ms"] = float(row["time_ms"])
            rec["mst_weight"] = int(row["mst_weight"])
            rec["mst_vertices"] = int(row["vertices"])
            rec["mst_edges"] = int(row["mst_edges"])
        except Exception as e:
            rec["status"] = "error"
            rec["note"] = str(e)
        bench.append(rec)
display(pd.DataFrame(bench))


,graph,status,time_ms,mst_weight,mst_vertices,mst_edges,note
0,amazon0601,ok,203.660,403387,403394,403387,
1,citationCiteseer,ok,364.821,268494,268495,268494,
2,as-skitter,ok,768.445,1695659,1696415,1695659,
3,coPapersDBLP,ok,1478.423,540485,540486,540485,
4,in-2004,ok,1905.952,1382774,1382908,1382774,
5,soc-LiveJournal1,ok,6157.704,4845695,4847571,4845695,
6,cit-Patents,ok,13516.957,3771141,3774768,3771141,
7,USA-road-d.USA,ok,26824.243,23947346,23947347,23947346,
8,delaunay_n24,ok,29203.740,16777215,16777216,16777215,
9,kron_g500-logn21,ok,29611.865,1544012,2097152,1543993,
